# AeroRAG-X grounded-agent GRPO on a Kaggle P100
This notebook first proves the GPU training path with synthetic fixtures. Point `CASES_PATH` at a real, versioned, protected-disjoint JSONL before treating a run as experiment evidence.

In [ ]:
!nvidia-smi
import subprocess

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
).strip()
print("GPU:", gpu)
assert "P100" in gpu, "Select GPU P100 in Kaggle notebook settings before continuing."

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/AeroRAG-X")
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/triasha72/AeroRAG-X.git", str(ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "torch==2.7.1",
        "torchvision==0.22.1",
        "torchaudio==2.7.1",
    ],
    check=True,
)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[llm,rl]"], check=True)
subprocess.run(["python", "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run(
    [
        "python",
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda); "
        "print(torch.cuda.get_device_name(0)); "
        "assert torch.cuda.get_device_capability(0) == (6, 0)",
    ],
    check=True,
)

In [ ]:
CASES_PATH = "data/training/grpo_grounded_agent_v0_1.template.jsonl"
CONFIG_PATH = "configs/grpo_kaggle_p100_smoke_v0_1.yaml"
OUTPUT_DIR = Path("/kaggle/working/aeroragx-grpo")
if CASES_PATH.endswith(".template.jsonl"):
    print("SMOKE TEST ONLY: these cases cannot establish model improvement.")
command = [
    "python",
    "scripts/train_grpo_grounded_agent_v0_1.py",
    "--cases",
    CASES_PATH,
    "--config",
    CONFIG_PATH,
    "--output-dir",
    str(OUTPUT_DIR),
]
subprocess.run(command, check=True)

In [ ]:
# Remove --execute for validation only. Add --resume after an interrupted session.
subprocess.run([*command, "--execute", "--resume"], check=True)

In [ ]:
import hashlib
import json
import tarfile

receipt = json.loads((OUTPUT_DIR / "run_receipt.json").read_text())
assert (OUTPUT_DIR / "final_adapter").is_dir()
archive = Path("/kaggle/working/aeroragx-grpo-output.tar.gz")
with tarfile.open(archive, "w:gz") as tar:
    tar.add(OUTPUT_DIR, arcname=OUTPUT_DIR.name)
print(json.dumps(receipt, indent=2))
print("archive_sha256:", hashlib.sha256(archive.read_bytes()).hexdigest())